# SimPO — Simple Preference Optimization
### Direct / Reward-Free Alignment  ·  Colab T4 (16 GB) ready

> **DPO/IPO/KTO all keep a reference model** (even if via adapter-disabling, it still costs a second forward pass every step). **ORPO** drops the reference but replaces it with an NLL/SFT term because it starts from a base model.
> **SimPO drops the reference model *and* adds no extra term at all** — the reward is just the model's own length-normalized log-probability, nothing else.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)
- **SimPO (Simple Preference Optimization with a Reference-Free Reward)** (*Meng, Xia & Chen, "SimPO: Simple Preference Optimization with a Reference-Free Reward"*) replaces DPO's **policy-vs-reference log-ratio reward** with the **length-normalized average log-probability** of the sequence itself:
  $$r_{\text{SimPO}}(x,y) = \frac{\beta}{|y|}\sum_{t=1}^{|y|}\log \pi_\theta(y_t \mid x, y_{<t})$$
- **No `π_ref` appears anywhere in this formula.** DPO's reward is a *ratio* to a frozen reference; SimPO's reward is the policy's **own per-token average log-likelihood** — literally the same quantity that governs how the model **scores its own generations at inference time** (length-normalized log-likelihood is the standard decoding-time metric).
- SimPO additionally bakes a **target reward margin `γ`** directly into the Bradley–Terry contrast, requiring the winning completion's reward to beat the losing one by **at least `γ`**, not merely by any positive amount.

### One-sentence definition of the mechanics
> **SimPO minimizes a Bradley–Terry classification loss over a reference-free, length-normalized average-log-probability reward with an explicit target margin γ, eliminating the reference model entirely while aligning the training-time reward with the same metric the policy uses to score its own generations at inference.**

The loss:
$$\mathcal{L}_{\text{SimPO}} = -\,\mathbb{E}_{(x,y_w,y_l)}\Big[\log \sigma\Big(\tfrac{\beta}{|y_w|}\log\pi_\theta(y_w|x) - \tfrac{\beta}{|y_l|}\log\pi_\theta(y_l|x) - \gamma\Big)\Big]$$

### The exact engineering problem it solves
- **Every reference-based method (DPO/IPO/KTO) pays a "second forward" tax.** Even with the adapter-disabling trick (one set of weights, adapters toggled), computing the reference log-probabilities still means **running the same batch through the network a second time** every step, plus the bookkeeping to toggle adapters on/off consistently.
- **SimPO removes the reference from the objective itself** — there is nothing to toggle, nothing frozen to keep in sync, no second forward pass of any kind. **One model, one forward pass per completion, done.**
- **It also fixes a DPO pathology in passing:** DPO's raw (non-length-normalized) log-probability reward structurally **favors shorter completions** during the log-ratio comparison and can be gamed by length — a documented failure mode. SimPO's **per-token average** removes the length confound directly, and because it is exactly the quantity used to score sequences at generation time, **training-time reward and inference-time behavior are the same metric** — closing a train/inference mismatch DPO has.

---

### The Human Element — Hugging Face datasets for SimPO

| HF path | What it is | Why it's shaped this way for SimPO |
|---|---|---|
| **`HuggingFaceH4/ultrafeedback_binarized`** (`train_prefs`) | The same GPT-4-scored `chosen`/`rejected` corpus used by the DPO/IPO notebooks. | SimPO needs the **same pairwise `(prompt, chosen, rejected)` schema** as DPO — reusing this corpus lets you A/B SimPO directly against DPO/IPO on **identical data**, isolating the effect of the reference-free, length-normalized objective. |
| **`princeton-nlp/llama3-ultrafeedback-armorm`** | The **official SimPO paper reproduction data** (Meng et al.'s released repo) — UltraFeedback prompts re-scored with the ArmoRM reward model for cleaner preference labels. | Built specifically for reproducing the paper's reported results on Instruct models; higher-quality reward-model labels reduce the noisy-pair problem that any reference-free method is more exposed to (no KL anchor to fall back on). Good next step once the pipeline below runs. |
| **`argilla/dpo-mix-7k`** | A small (~7k) curated preference mix. | Same schema, far fewer pairs — useful for a **fast T4 sweep** of `beta`/`simpo_gamma` before committing to a longer run. |

**Why the schema is identical to DPO's:** SimPO changes only **what the reward is** (length-normalized log-prob vs. a reference log-ratio), not the data contract. It still needs `(prompt, chosen, rejected)` triples sharing a prompt — the margin is meaningless without two completions to contrast.

> This notebook trains on **`HuggingFaceH4/ultrafeedback_binarized` / `train_prefs`**, starting from the **Instruct** checkpoint (Section 3).

---

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the mathematical reason SimPO works
- **DPO's derivation keeps a `π_ref` term because it starts from a KL-constrained RL objective** — the closed-form optimum is expressed *relative to* a reference, and the reward is that ratio. SimPO **doesn't derive from that objective at all**; it starts from the observation that at **decoding time**, a model's own **length-normalized log-likelihood** is the quantity that actually governs generation quality (it's what beam search / likelihood-based sampling implicitly optimizes). SimPO simply **uses that quantity as the reward directly** — no reference needed because nothing is being measured *relative to* anything else.
- **Why this doesn't blow up like DPO's unbounded reward (the IPO problem):** each token's log-probability is **≤ 0** by definition, so the **average log-probability reward is upper-bounded at 0** — a completion can never have a reward above "the model is 100% certain of every token." This structural ceiling is what keeps the margin from diverging the way DPO's un-normalized log-ratio can; SimPO gets a form of boundedness **for free from the reward's own definition**, without IPO's explicit regression target or DPO/KTO's reference-anchored KL term.
- **The target margin `γ` is what makes "positive separation" into "confident separation."** Without it, the Bradley–Terry loss is satisfied by *any* margin greater than zero, including tiny, noise-level ones. `γ` forces the winning completion's reward to clear the losing one by a **fixed amount**, which the paper found substantially improves downstream generation quality.

#### VRAM & Compute Impact
- **The lightest of all four methods in this folder.** Like ORPO, there is **no reference model at all** — but unlike ORPO, SimPO adds **no second loss term** (no NLL/SFT anchor), because it assumes the **Instruct** checkpoint is already a good starting policy (same assumption as DPO/IPO/KTO).
- **Two forward passes per step** (chosen + rejected), **both through the same trainable policy** — no reference forward, no adapter toggling, no frozen copy to keep in sync.
- **Lower training budget than ORPO:** ORPO needs extra LoRA capacity, more epochs, and more data because it is *also* learning instruction-following from a base model. SimPO, starting from Instruct, needs only a **light preference-only pass** — smaller LoRA rank, fewer epochs, comparable to DPO/IPO/KTO's training budget.
- **Net:** SimPO combines **ORPO's zero-reference memory profile** with **DPO's light, short preference-only training schedule** — the cheapest combination of the four.

#### Pros & Cons

**Pros**
- **No reference model, period** — not even the adapter-disabling trick is needed; nothing to toggle, nothing to keep frozen in sync.
- **Fixes DPO's length bias** — the length-normalized reward removes the incentive to game the objective via completion length.
- **Reward matches inference-time behavior** — the training objective and the metric that governs actual generation are the same quantity.
- **Simplest loss of the four** — one term, no auxiliary NLL/BC regularizer, no reference bookkeeping.

**Cons**
- **No explicit anchor of any kind** — not a KL-to-reference (DPO/IPO/KTO) and not an NLL-to-chosen (ORPO). Stability depends entirely on starting from a good Instruct checkpoint and conservative optimization (small LR, LoRA, short runs); there is nothing to fall back on if training drifts.
- **Two more interacting hyperparameters** — `beta` and `simpo_gamma` must be tuned together; the paper sweeps `beta ∈ {2.0, 2.5}` and a `γ/β` ratio, and values don't transfer from DPO's `beta`.
- **Still off-policy** — learns only from the fixed pairs; no exploration (PPO's advantage).
- **More exposed to noisy pairs** than reference-anchored methods, since there is no KL term to limit how far a bad label can push the policy.

#### Metrics to watch (TRL's `CPOTrainer` logs the same `rewards/*` keys)
- **`rewards/chosen`, `rewards/rejected`** — now literally `(β/|y|)·log p(y)` per side (not a ratio to anything) — small negative numbers, **bounded above by 0**.
- **`rewards/margins`** — should grow to **exceed the configured `simpo_gamma`**, not just clear zero; that's the target-margin mechanism working.
- **`rewards/accuracies`** — fraction of pairs with positive margin; should climb toward 1.0.
- **No `nll_loss`** — unlike ORPO, there is no SFT/NLL term to watch; if you see one logged, `cpo_alpha` wasn't set to 0 and you're not running pure SimPO.

---

## 3. Production-Grade Implementation (Colab T4, 16 GB)

**QLoRA (4-bit) + TRL `CPOTrainer` with `loss_type="simpo"`, `cpo_alpha=0.0` — no reference model, no NLL term.**

> ⚙️ **SimPO lives inside `CPOTrainer`:** TRL implements SimPO as a special case of CPO (Contrastive Preference Optimization) — set **`loss_type="simpo"`** and **`cpo_alpha=0.0`** to disable CPO's NLL/behavior-cloning term entirely, leaving pure SimPO. Note `CPOTrainer`'s constructor **doesn't even accept a `ref_model` argument** — reference-free isn't an option here, it's the only mode.

**Executable pipeline:**

| Step | What | Notes |
|---|---|---|
| 1 | 4-bit `Qwen2.5-0.5B-Instruct` + LoRA + tokenizer | **one policy, no reference at all** |
| 2 | `HuggingFaceH4/ultrafeedback_binarized` → `(prompt, chosen, rejected)` | same schema as DPO/IPO |
| 3 | `CPOConfig(loss_type="simpo", cpo_alpha=0.0, beta, simpo_gamma)` | length-normalized reward + target margin |
| 4 | `CPOTrainer.train()` | single reference-free loss |
| 5 | Save adapter · export · inference | ship it |

### Environment Setup

In [ ]:
%pip install torchao==0.16.0 transformers trl peft accelerate bitsandbytes datasets

In [ ]:
import torch, os, math
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig
from trl.experimental.cpo import CPOTrainer, CPOConfig  # SimPO = CPOTrainer with loss_type="simpo", cpo_alpha=0.0

set_seed(42)
# Reduce CUDA fragmentation OOMs on the T4.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# ---- Hardware-aware dtype selection (do NOT hardcode bf16 on a T4) ----------------
# The Colab T4 is Turing (compute capability 7.5). bfloat16 TENSOR CORES only exist on
# Ampere (SM 8.0) and newer. Forcing bf16 here makes the bitsandbytes 4-bit dequant path
# return garbage, which becomes NaN logits -> NaN softmax -> "CUDA error: device-side
# assert triggered" inside torch.multinomial at generation time. Detect, don't assume.
major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8                                   # True on A100/L4/H100, False on T4
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print(f"device capability sm_{major}{_minor} | bf16 usable: {USE_BF16} | compute dtype: {COMPUTE_DTYPE}")

### Step 1 — Quantization, LoRA & Tokenizer (from the **Instruct** checkpoint, NO reference)

Like DPO/IPO/KTO — **not** ORPO — we start from the **Instruct** checkpoint, because SimPO assumes SFT is already done. Unlike DPO/IPO/KTO, there is **no reference-model machinery at all**: no adapter-disabling trick, no second forward — this LoRA config is used for plain, single-model fine-tuning.

In [ ]:
# The SFT/Instruct start point. SimPO assumes SFT is already done (same assumption as DPO/IPO/KTO).
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 4-bit NF4. Only ONE model, ever — SimPO never runs a reference forward pass.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # fp16 on T4 (Turing has no bf16 tensor cores)
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # pads batched chosen/rejected; Qwen needs a pad id

# Light preference-only LoRA (same rank as DPO/IPO/KTO) — SimPO is NOT doing SFT from base,
# so it does not need ORPO's higher-capacity r=32 setup.
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

policy_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",  # mem-efficient attention (T4 has no FlashAttn-2)
)
policy_model.config.use_cache = False  # required with gradient checkpointing
print(policy_model.get_memory_footprint() / 1e9, "GB base (4-bit)  — and NO reference model, ever")

### Step 2 — Dataset: `HuggingFaceH4/ultrafeedback_binarized`

Same pairwise schema and same corpus the DPO/IPO notebooks use — flattened to the standard `(prompt, chosen, rejected)` triple so a direct comparison against DPO/IPO isolates the effect of SimPO's objective alone.

In [ ]:
raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
raw = raw.shuffle(seed=42).select(range(1000))  # subset so a T4 finishes in a reasonable time

def to_simpo_triple(ex):
    # ex["chosen"] = [{user}, {assistant}] (single-turn). Everything but the final assistant
    # message is the SHARED prompt.
    prompt_msgs = ex["chosen"][:-1]
    return {
        "prompt":   tokenizer.apply_chat_template(prompt_msgs, tokenize=False,
                                                  add_generation_prompt=True),
        "chosen":   ex["chosen"][-1]["content"],    # preferred completion (raw text)
        "rejected": ex["rejected"][-1]["content"],  # dispreferred completion (raw text)
    }

# String prompt/chosen/rejected => TRL treats it as STANDARD format (no re-templating).
simpo_ds = raw.map(to_simpo_triple, remove_columns=raw.column_names)
print(simpo_ds)

In [ ]:
# Step 2.5 — Free leftover GPU memory (run before training)

import gc
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1e9:.2f} GB / {total/1e9:.2f} GB  (expect several GB free before training)")

### Step 3 — `CPOConfig(loss_type="simpo")` & `CPOTrainer`

**`beta` and `simpo_gamma` are the two SimPO knobs** — the paper sweeps `beta ∈ {2.0, 2.5}` (notably **higher** than DPO's typical `0.1`, since it scales a *bounded* average-log-prob reward rather than an unbounded log-ratio) and a `γ/β` ratio, so `simpo_gamma` is commonly a fraction of `beta`. **`cpo_alpha=0.0` is what makes this pure SimPO** rather than CPO — it disables the NLL/behavior-cloning term entirely.

In [ ]:
"""
- Compute warmup steps dynamically

`warmup_steps` is a step count, not a ratio — derive it from the actual optimizer-step total
(`rows * epochs / (batch * grad_accum)`) so it scales automatically if you change the subset size,
epoch count, or batch settings above. **~10% warmup** is the standard preference-tuning default.
"""

per_device_train_batch_size = 1
gradient_accumulation_steps = 8
num_train_epochs = 1

# Optimizer steps = passes through the data / effective batch size (world_size=1 on a single T4).
total_optimizer_steps = math.ceil(len(simpo_ds) * num_train_epochs / (per_device_train_batch_size * gradient_accumulation_steps))
warmup_steps = max(1, math.ceil(total_optimizer_steps * 0.1))  # ~10% warmup
print(f"optimizer steps: {total_optimizer_steps}  ->  warmup_steps: {warmup_steps}")

In [ ]:
simpo_config = CPOConfig(
    output_dir="./simpo_output",
    run_name="simpo-t4",

    # ---- The SimPO knobs ----
    loss_type="simpo",   # length-normalized avg-logprob reward + target margin (reference-free)
    cpo_alpha=0.0,        # disables CPO's NLL/behavior-cloning term -> PURE SimPO (paper setting)
    beta=2.0,             # SimPO beta scales a BOUNDED reward -> paper uses higher values than DPO's 0.1
    simpo_gamma=0.5,      # target reward margin; paper sweeps gamma/beta ratios around 0.3-0.5

    # ---- Sequence budget (chosen AND rejected tokenized) ----
    max_length=512,        # cap on prompt + completion
    max_prompt_length=256,  # prompt-only cap

    # ---- T4 16 GB hardening ----
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,  # effective batch = 8
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=USE_BF16, fp16=not USE_BF16,  # match the GPU: fp16 on T4, bf16 on Ampere+
    optim="paged_adamw_8bit",

    # ---- Optimization schedule ----
    learning_rate=5e-6,          # small LR; no reference/NLL anchor to catch a bad step
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    num_train_epochs=num_train_epochs,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

# NO ref_model argument — CPOTrainer's signature does not accept one. SimPO is reference-free
# by construction, not by an optional flag.
simpo_trainer = CPOTrainer(
    model=policy_model,
    args=simpo_config,
    train_dataset=simpo_ds,
    processing_class=tokenizer,
    peft_config=peft_config,
)

### Step 4 — Train

Watch **`rewards/margins` exceed `simpo_gamma`** (0.5 here) — that clearance, not just a positive margin, is what the target-margin term is enforcing.

In [ ]:
simpo_trainer.train()

# Save the aligned LoRA adapter (a few MB, not GB).
simpo_trainer.save_model("./simpo_aligned_adapter")
tokenizer.save_pretrained("./simpo_aligned_adapter")
print("Saved -> ./simpo_aligned_adapter")

In [ ]:
# Sanity-check the trained weights before saving/inference.
# SimPO has NO reference or NLL anchor (see [Context Block] Cons) -- a bad optimizer step
# has nothing pulling it back, so a diverged run shows up as NaN/Inf LoRA weights. Left
# unchecked, that surfaces later as a cryptic "CUDA error: device-side assert triggered"
# inside generate()'s sampling -- and once THAT fires, the CUDA context is poisoned and
# nothing works until you restart the runtime. Catch it here instead, as a clear message.
bad_params = [
    n for n, p in policy_model.named_parameters()
    if p.requires_grad and (torch.isnan(p).any() or torch.isinf(p).any())
]
if bad_params:
    raise RuntimeError(
        f"Training diverged: {len(bad_params)} LoRA tensor(s) contain NaN/Inf "
        f"(e.g. {bad_params[0]}). Restart the runtime, then retrain with a lower "
        "learning_rate (try 1e-6) and/or lower beta (try 1.0) -- SimPO has no "
        "reference/KL anchor to recover from an unstable step."
    )
print("Trained weights OK: no NaN/Inf found.")

## Export — Download the Aligned Adapter (Optional)

In [ ]:
import shutil, os

folder_to_zip = "./simpo_aligned_adapter"
output_filename = "simpo_aligned_adapter.zip"

shutil.make_archive(output_filename.replace(".zip", ""), "zip", folder_to_zip)
if os.path.exists(output_filename):
    print(f"File: {output_filename}  ({os.path.getsize(output_filename)/1e6:.2f} MB)")
else:
    print("Zip not found — run training + save first.")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found. Did training + zipping finish?")

---

## Model Usage — Evaluate the SimPO-Aligned Policy

Reload the **Instruct base + SimPO adapter** and generate with the **same chat template** used in training.

In [ ]:
# IMPORTANT: free the TRAINING model before loading a second copy for inference.
# The trainer + policy_model are still resident here; building another 4-bit model on top
# of them is what tips a 16 GB T4 over. A failure inside that load leaves the CUDA context
# in a broken state, which then surfaces asynchronously as "device-side assert triggered"
# at the next CUDA call -- even one as trivial as inputs.to(device).
import gc, torch

for _obj in ["simpo_trainer", "policy_model"]:
    if _obj in globals():
        del globals()[_obj]
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free before inference load: {free/1e9:.2f} GB / {total/1e9:.2f} GB")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Re-derive the dtype here so this cell works standalone after a restart.
major, _minor = torch.cuda.get_device_capability()
USE_BF16 = major >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "./simpo_aligned_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # fp16 on T4 (Turing has no bf16 tensor cores)
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
model = PeftModel.from_pretrained(base, adapter_path)  # attach the SimPO adapter
model.eval()

# Guard against the classic device-side-assert cause: a token id outside the embedding
# table. If these ever disagree, generation asserts inside the embedding lookup.
print("embedding rows:", model.get_input_embeddings().weight.shape[0])
print("tokenizer len :", len(tokenizer), "| pad id:", tokenizer.pad_token_id, "| eos id:", tokenizer.eos_token_id)

In [ ]:
# Same chat template the SimPO loop used (single user turn).
def generate_response(user_prompt, max_new_tokens=256, temperature=0.7, do_sample=True):
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")

    # Fail fast, on CPU, with a READABLE error if any id is outside the embedding table.
    vocab_rows = model.get_input_embeddings().weight.shape[0]
    max_id = int(inputs["input_ids"].max())
    if max_id >= vocab_rows:
        raise ValueError(f"token id {max_id} >= embedding rows {vocab_rows} (tokenizer/model mismatch)")

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # One forward pass to verify the logits are finite BEFORE sampling. torch.multinomial
    # raises an opaque "device-side assert triggered" on NaN/Inf input and poisons the CUDA
    # context; this turns that into a plain Python error naming the real cause.
    with torch.no_grad():
        probe = model(**inputs).logits
    if not torch.isfinite(probe).all():
        raise RuntimeError(
            "Model produced non-finite logits. Almost always a dtype/hardware mismatch: "
            "bf16 on a Turing T4 (use COMPUTE_DTYPE), or a diverged training run."
        )

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature if do_sample else None,
            top_p=0.9 if do_sample else None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated completion (slice off the prompt tokens).
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
test_prompts = [
    "Why does SimPO not need a reference model at all, unlike DPO?",
    "My friend has been feeling really down lately. How can I support them?",
]

print("--- SimPO-Aligned Responses ---")
for i, p in enumerate(test_prompts, 1):
    print(f"\n[Prompt {i}]: {p}")
    print(f"[Response]: {generate_response(p).strip()}")
    print("-" * 60)